## Logistic regression

In [1]:
import pandas as pd
pd.options.mode.chained_assignment = None
import pylab as pl
import numpy as np
import scipy.optimize as opt
from sklearn import preprocessing
%matplotlib inline 
import matplotlib.pyplot as plt

In [2]:
# Load data from Excel file
patients_data = pd.read_csv("Texture features.csv", index_col=0)
patients_data.head()

,INFO_PatientID,INFO_ProcessDateOfTexture,INFO_SeriesDate,INFO_Series,INFO_ActualFrameDuration,INFO_NameOfRoi,CONVENTIONAL_SUVbwmin,CONVENTIONAL_SUVbwmean,CONVENTIONAL_SUVbwstd,CONVENTIONAL_SUVbwmax,...,GLZLM_HGZE,GLZLM_SZLGE,GLZLM_SZHGE,GLZLM_LZLGE,GLZLM_LZHGE,GLZLM_GLNU,GLZLM_ZLNU,GLZLM_ZP,TimePosition,zLocation[onlyFor2DROI]
INFO_PatientName,,,,,,,,,,,,,,,,,,,,,
LIFEX,LIFEx,Sun Jan 23 18:19:19 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,0.0,0.108004,0.319234,0.064479,0.476722,...,2.500000,0.000192,0.000585,2107.625000,11034.500000,1.000000,1.000000,0.016129,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 18:19:20 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,1.0,2.910082,5.854704,1.387413,8.991810,...,363.808824,0.002428,208.152420,0.021055,3025.955882,4.323529,25.000000,0.482270,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 18:38:58 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,0.0,0.188521,0.323404,0.057192,0.449061,...,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 18:38:59 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,1.0,3.238746,5.991053,1.307934,8.991810,...,392.298246,0.001739,206.457640,0.021977,3172.052632,4.228070,16.614035,0.438462,0.0,NaN
LIFEX,LIFEx,Sun Jan 23 18:51:51 AST 2022,2018-04-13,2006_PET TAP 6I8SG5 LIFEx_PT,3.000000142492354 min,0.0,0.188521,0.323404,0.057192,0.449061,...,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0,NaN


### Data preprocessing and Feature selection

In [3]:
# Selecting features using Decision Trees

# check missing values in variables
patients_data.isnull().sum()

INFO_PatientID               0
INFO_ProcessDateOfTexture    0
INFO_SeriesDate              0
INFO_Series                  0
INFO_ActualFrameDuration     0
                            ..
GLZLM_GLNU                   0
GLZLM_ZLNU                   0
GLZLM_ZP                     0
TimePosition                 0
zLocation[onlyFor2DROI]      8
Length: 101, dtype: int64

In [4]:
# Defining X (Features, so we drop INFO_NameOfRoi as well as irrelevent columns)
X = patients_data.drop(['INFO_NameOfRoi', 'INFO_PatientID', 'INFO_ProcessDateOfTexture' , 'INFO_SeriesDate' , 'INFO_Series' , 'INFO_ActualFrameDuration'], axis=1)
X.head()

,CONVENTIONAL_SUVbwmin,CONVENTIONAL_SUVbwmean,CONVENTIONAL_SUVbwstd,CONVENTIONAL_SUVbwmax,CONVENTIONAL_SUVbwQ1,CONVENTIONAL_SUVbwQ2,CONVENTIONAL_SUVbwQ3,CONVENTIONAL_SUVbwSkewness,CONVENTIONAL_SUVbwKurtosis,CONVENTIONAL_SUVbwExcessKurtosis,...,GLZLM_HGZE,GLZLM_SZLGE,GLZLM_SZHGE,GLZLM_LZLGE,GLZLM_LZHGE,GLZLM_GLNU,GLZLM_ZLNU,GLZLM_ZP,TimePosition,zLocation[onlyFor2DROI]
INFO_PatientName,,,,,,,,,,,,,,,,,,,,,
LIFEX,0.108004,0.319234,0.064479,0.476722,0.283809,0.321648,0.372574,-0.502561,3.427661,0.427661,...,2.500000,0.000192,0.000585,2107.625000,11034.500000,1.000000,1.000000,0.016129,0.0,NaN
LIFEX,2.910082,5.854704,1.387413,8.991810,4.692568,5.716900,6.903404,0.116157,2.321317,-0.678683,...,363.808824,0.002428,208.152420,0.021055,3025.955882,4.323529,25.000000,0.482270,0.0,NaN
LIFEX,0.188521,0.323404,0.057192,0.449061,0.283840,0.328323,0.365392,-0.161892,2.349181,-0.650819,...,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0,NaN
LIFEX,3.238746,5.991053,1.307934,8.991810,5.146196,5.923580,6.928142,0.150938,2.437242,-0.562758,...,392.298246,0.001739,206.457640,0.021977,3172.052632,4.228070,16.614035,0.438462,0.0,NaN
LIFEX,0.188521,0.323404,0.057192,0.449061,0.283840,0.328323,0.365392,-0.161892,2.349181,-0.650819,...,2.000000,0.333484,0.333739,1242.000000,7367.000000,1.666667,1.000000,0.024793,0.0,NaN


In [5]:
# Defining y (two classes: tumorous, non-tumorous)
y = np.asarray(patients_data[['INFO_NameOfRoi']])
y[0:9]

array([[0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.]])

In [7]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
dt = DecisionTreeClassifier(max_depth=3)
dt.fit(X,y) # fit the model
# plot the tree
plt.figure(figsize=(12,8))
from sklearn import tree
tree.plot_tree(dt.fit(X,y))

ValueError: could not convert string to float: '0.10800401121377945|0.10800401121377945|0.2369452863931656|'

### Testing and Training Split and Normalization

In [ ]:
from sklearn.model_selection import train_test_split
# 20% test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)
print ('Train set:', X_train.shape,  y_train.shape)
print ('Test set:', X_test.shape,  y_test.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=4)
print ('Train set:', X_train.shape,  y_train.shape)
print ('Validation set:', X_val.shape,  y_val.shape)

In [ ]:
# Normalizing the train, test, and validate dataset separately to avoid data leakage from the test data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

### Modeling Logistic Regression with Scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(C=0.01, solver='liblinear').fit(X_train,y_train.ravel())
clf

In [ ]:
# Predicting using test set
y_2 = clf.predict(X_test)
y_2

### Evaluation metrics

In [ ]:
# The first column is the probability of class 0, P(Y=0|X), and second column is probability of class 1, P(Y=1|X)
y_2_prob = clf.predict_proba(X_test)
y_2_prob

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report,plot_confusion_matrix, plot_roc_curve,plot_precision_recall_curve

In [ ]:
def model_evaluation(model):
    
    print(f'Classification Report :\n {classification_report(y_test,y_pred)}')
    fig, ax = plt.subplots(dpi=100);
    print(f'Confusion Matrix :\n {plot_confusion_matrix(model,X_test,y_test,ax=ax)}')
    print(f'ROC Curve :\n {plot_roc_curve(model,X_test,y_test)}')
    print(f'Precision Recall Curve :\n {plot_precision_recall_curve(model,X_test,y_test)}')

In [ ]:
y_pred = clf.predict(X_test)
model_evaluation(clf)